# EDA and Preprocessing

In this notebook, we perform exploratory data analysis and preprocessing for the first anomaly detection experiment.

Training baseline:
- `Monday-WorkingHours.pcap_ISCX.csv`
- Contains only BENIGN traffic

Evaluation data:
- `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv`
- Contains BENIGN and DDoS traffic

The goal is to prepare clean numerical features for unsupervised anomaly detection.

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("Project root:", Path.cwd())

Project root: D:\Development\Python Projects\network-traffic-anomaly-detection


In [2]:
import pandas as pd
import numpy as np

In [3]:
data_dir = Path("data/raw/network-intrusion-dataset/versions/1")

train_file = data_dir / "Monday-WorkingHours.pcap_ISCX.csv"
test_file = data_dir / "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain labels:")
print(train_df["Label"].value_counts())

print("\nTest labels:")
print(test_df["Label"].value_counts())

Train shape: (529918, 79)
Test shape: (225745, 79)

Train labels:
Label
BENIGN    529918
Name: count, dtype: int64

Test labels:
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


In [8]:
#Basic Inspection

train_df.head()



,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [9]:
test_df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [10]:
train_df.columns.tolist()

['Destination Port',
 'Flow Duration',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Total Length of Fwd Packets',
 'Total Length of Bwd Packets',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min',
 'Fwd IAT Total',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Fwd IAT Max',
 'Fwd IAT Min',
 'Bwd IAT Total',
 'Bwd IAT Mean',
 'Bwd IAT Std',
 'Bwd IAT Max',
 'Bwd IAT Min',
 'Fwd PSH Flags',
 'Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'Fwd Header Length',
 'Bwd Header Length',
 'Fwd Packets/s',
 'Bwd Packets/s',
 'Min Packet Length',
 'Max Packet Length',
 'Packet Length Mean',
 'Packet Length Std',
 'Packet Length Variance',
 'FIN Flag Count',
 'SYN Flag Count',
 'RST Flag Count',
 'PSH Flag Count',
 'ACK Flag Count',
 'UR

In [11]:
same_columns = list(train_df.columns) == list(test_df.columns)
print("Same columns:", same_columns)

Same columns: True


In [12]:
train_df.dtypes.value_counts()

int64      54
float64    24
object      1
Name: count, dtype: int64

In [13]:
non_numeric_cols = train_df.select_dtypes(exclude=[np.number]).columns.tolist()
non_numeric_cols

['Label']

In [14]:
#missing values

print("Train missing values:", train_df.isna().sum().sum())
print("Test missing values:", test_df.isna().sum().sum())


Train missing values: 64
Test missing values: 4


In [15]:
#infinte values
train_numeric = train_df.select_dtypes(include=[np.number])
test_numeric = test_df.select_dtypes(include=[np.number])

print("Train infinite values:", np.isinf(train_numeric).sum().sum())
print("Test infinite values:", np.isinf(test_numeric).sum().sum())

Train infinite values: 810
Test infinite values: 64


In [16]:
#duplicate rows
print("Train duplicate rows:", train_df.duplicated().sum())
print("Test duplicate rows:", test_df.duplicated().sum())

Train duplicate rows: 26935
Test duplicate rows: 2633


In [17]:
# Columns with missing values in train data
train_missing_cols = train_df.isna().sum()
train_missing_cols = train_missing_cols[train_missing_cols > 0].sort_values(ascending=False)

print("Train columns with missing values:")
print(train_missing_cols)

# Columns with missing values in test data
test_missing_cols = test_df.isna().sum()
test_missing_cols = test_missing_cols[test_missing_cols > 0].sort_values(ascending=False)

print("\nTest columns with missing values:")
print(test_missing_cols)

Train columns with missing values:
Flow Bytes/s    64
dtype: int64

Test columns with missing values:
Flow Bytes/s    4
dtype: int64


In [18]:
# Select numeric columns
train_numeric = train_df.select_dtypes(include=[np.number])
test_numeric = test_df.select_dtypes(include=[np.number])

# Columns with infinite values in train data
train_inf_cols = np.isinf(train_numeric).sum()
train_inf_cols = train_inf_cols[train_inf_cols > 0].sort_values(ascending=False)

print("Train columns with infinite values:")
print(train_inf_cols)

# Columns with infinite values in test data
test_inf_cols = np.isinf(test_numeric).sum()
test_inf_cols = test_inf_cols[test_inf_cols > 0].sort_values(ascending=False)

print("\nTest columns with infinite values:")
print(test_inf_cols)

Train columns with infinite values:
Flow Packets/s    437
Flow Bytes/s      373
dtype: int64

Test columns with infinite values:
Flow Packets/s    34
Flow Bytes/s      30
dtype: int64


### Missing and Infinite Value Analysis

The missing and infinite values are concentrated in the rate-based features `Flow Bytes/s` and `Flow Packets/s`. These features are calculated using flow duration, so undefined or infinite values can occur when the flow duration is zero.

Since the number of affected rows is very small compared to the dataset size, the preprocessing strategy is to replace infinite values with missing values and then remove rows containing missing values. This avoids introducing artificial values into rate-based traffic features.

In [19]:
#clean invalid values and duplicates

train_clean = train_df.copy()
test_clean = test_df.copy()

# Replace positive and negative infinity with NaN
train_clean = train_clean.replace([np.inf, -np.inf], np.nan)
test_clean = test_clean.replace([np.inf, -np.inf], np.nan)

# Drop rows with missing values
train_clean = train_clean.dropna()
test_clean = test_clean.dropna()

# Remove duplicate rows
train_clean = train_clean.drop_duplicates()
test_clean = test_clean.drop_duplicates()

print("Original train shape:", train_df.shape)
print("Clean train shape:", train_clean.shape)

print("\nOriginal test shape:", test_df.shape)
print("Clean test shape:", test_clean.shape)

print("\nTrain labels after cleaning:")
print(train_clean["Label"].value_counts())

print("\nTest labels after cleaning:")
print(test_clean["Label"].value_counts())

Original train shape: (529918, 79)
Clean train shape: (502650, 79)

Original test shape: (225745, 79)
Clean test shape: (223082, 79)

Train labels after cleaning:
Label
BENIGN    502650
Name: count, dtype: int64

Test labels after cleaning:
Label
DDoS      128014
BENIGN     95068
Name: count, dtype: int64


In [20]:
train_numeric_clean = train_clean.select_dtypes(include=[np.number])
test_numeric_clean = test_clean.select_dtypes(include=[np.number])

print("Train missing values after cleaning:", train_clean.isna().sum().sum())
print("Test missing values after cleaning:", test_clean.isna().sum().sum())

print("Train infinite values after cleaning:", np.isinf(train_numeric_clean).sum().sum())
print("Test infinite values after cleaning:", np.isinf(test_numeric_clean).sum().sum())

print("Train duplicate rows after cleaning:", train_clean.duplicated().sum())
print("Test duplicate rows after cleaning:", test_clean.duplicated().sum())

Train missing values after cleaning: 0
Test missing values after cleaning: 0
Train infinite values after cleaning: 0
Test infinite values after cleaning: 0
Train duplicate rows after cleaning: 0
Test duplicate rows after cleaning: 0


## Cleaning Invalid Rows

The initial data quality checks showed that the selected training and test datasets contain missing values, infinite values, and duplicate rows. These issues must be handled before applying anomaly detection models, because most machine learning algorithms cannot process `NaN` or infinite values.

The missing and infinite values were concentrated in the rate-based features:

- `Flow Bytes/s`
- `Flow Packets/s`

These features are calculated using the flow duration. Therefore, if a flow has zero or extremely small duration, the rate calculation can become undefined or infinite.

Since the number of affected rows is very small compared to the total dataset size, we remove these invalid rows instead of imputing artificial values. This keeps the traffic-rate features realistic and avoids introducing values that may distort the anomaly detection model.

The cleaning process is:

1. Replace positive and negative infinity values with `NaN`.
2. Drop all rows containing `NaN` values.
3. Remove duplicate rows.

After cleaning, the dataset contains no missing values, no infinite values, and no duplicate rows.

## Separate features and lables

In [21]:
# Separate labels
y_train = train_clean["Label"]
y_test = test_clean["Label"]

# Separate features
X_train = train_clean.drop(columns=["Label"])
X_test = test_clean.drop(columns=["Label"])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (502650, 78)
X_test shape: (223082, 78)
y_train shape: (502650,)
y_test shape: (223082,)


In [22]:
# Convert labels to binary: BENIGN = 0, attack = 1
y_test_binary = y_test.apply(lambda x: 0 if x == "BENIGN" else 1)

print(y_test_binary.value_counts())

Label
1    128014
0     95068
Name: count, dtype: int64


In [23]:
non_numeric_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Non-numeric feature columns:", non_numeric_features)

Non-numeric feature columns: []


### Scale the features

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (502650, 78)
X_test_scaled shape: (223082, 78)


In [25]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled.describe().T[["mean", "std"]].head(10)

,mean,std
Destination Port,-5.733535e-17,1.000001
Flow Duration,1.311815e-17,1.000001
Total Fwd Packets,-4.240780e-20,1.000001
Total Backward Packets,2.572740e-18,1.000001
Total Length of Fwd Packets,3.166449e-18,1.000001
Total Length of Bwd Packets,-4.735538e-19,1.000001
Fwd Packet Length Max,5.021084e-17,1.000001
Fwd Packet Length Min,3.731887e-17,1.000001
Fwd Packet Length Mean,-2.316031e-16,1.000001
Fwd Packet Length Std,2.171280e-16,1.000001


## Feature Scaling

The feature values in the dataset have very different ranges. For example, duration-based and byte-rate features can have much larger values than flag-count features. Since distance-based anomaly detection methods such as DBSCAN and neural methods such as autoencoders are sensitive to feature scale, the numerical features are standardized before modeling.

The scaler is fitted only on the training baseline data and then applied to the test data. This prevents information from the evaluation data from leaking into the preprocessing step.

## Scaling Verification

After applying `StandardScaler`, the training features have means close to 0 and standard deviations close to 1. Very small values such as `1e-17` are caused by floating-point precision and can be interpreted as approximately zero.

This confirms that the feature scaling step was successful. The scaled training and test feature matrices are now ready for anomaly detection models.

In [26]:
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

X_train_scaled.to_csv(processed_dir / "X_train_scaled.csv", index=False)
X_test_scaled.to_csv(processed_dir / "X_test_scaled.csv", index=False)

y_test_binary.to_csv(processed_dir / "y_test_binary.csv", index=False)

print("Processed files saved to:", processed_dir.resolve())

Processed files saved to: D:\Development\Python Projects\network-traffic-anomaly-detection\data\processed


In [27]:
y_test.to_csv(processed_dir / "y_test_labels.csv", index=False)

## Preprocessing Summary

The selected training and evaluation datasets were cleaned and prepared for anomaly detection. Infinite values were replaced with missing values, invalid rows were removed, and duplicate rows were dropped. The label column was separated from the feature matrix, and the evaluation labels were converted into binary form where `0` represents BENIGN traffic and `1` represents attack traffic.

The numerical features were standardized using `StandardScaler`. The scaler was fitted only on the training baseline data and then applied to the evaluation data to avoid data leakage.

The processed feature matrices and labels were saved for use in the anomaly detection modeling notebook.